# E2.8 · Auditability of autonomous action

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.7 · Documentation that survives supervision](https://spbreed.github.io/cyber-commons/lessons/E2.7.html)**.

| | |
|---|---|
| Tools used | Keycloak |

## What this lesson is

**What it covers.** Produce an audit trail from the A2 chain that names authority at every hop.

**Why a security engineer needs it.** No trail showing under whose authority the agent acted. The control it builds is: the delegation chain *is* the audit trail.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

"Why did it do that?" is a question with a legal deadline attached. Logging designed forwards records what was convenient; logging designed backwards from the auditor's question records what is needed.

> **At CyberTravels.** “Why did CyberTravels issue that refund?” has a legal deadline attached. Logging designed backwards from that question records the booking note; logging designed forwards records the HTTP call. R11.

## 2 · The framework

```
   design the log backwards from the question

   auditor asks            log must contain
   why this decision?  ->  the input that motivated it
   on whose behalf?    ->  principal + delegation chain
   under what rule?    ->  policy version at that moment
   who reviewed it?    ->  approver identity and what they saw

   forwards-designed logs record what was convenient
```

Auditability of autonomous action reduces to one question:

> For any single action, can you produce **who caused it** and **what they were
> allowed to do**?

Answering it needs two capabilities that must both be present at the moment the
action happens, because neither can be reconstructed afterwards:

- **Attribution** — the acting identity, the principal, and the chain between
  them (A2.5, EV-1).
- **Replay** — the prompts, the tool results, the pinned model version and the
  seed (D2.5).

Attribution without replay tells you who acted but not why. Replay without
attribution tells you what happened but not on whose authority. Regulators and
auditors ask both, usually in that order.

## 3 · The procedure, as a skill

A record can name the acting identity, the principal, the chain and the scopes, be internally consistent, and be false. The skill scores three cases and adds the check that separates delegation from impersonation.

### The skill — [`skills/regulatory/autonomous-action-auditability/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/regulatory/autonomous-action-auditability/SKILL.md)

```yaml
name: autonomous-action-auditability
description: >-
  Decide whether an audit record makes an autonomous action answerable, and
  demonstrate a record that is complete, internally consistent and false. Use
  when an audit trail is being accepted as evidence of what an agent did.
allowed-tools: Read, Grep, Glob
```

# Complete, consistent, and false

Auditability of autonomous action is not "is there a record". A record can name
the acting identity, the principal, the delegation chain and the scopes, be
internally consistent, and still be false — because the token it rests on was
impersonated rather than delegated. Answerable requires the record *and* a way
to tell those apart.

## When to use this

When designing audit records for agent actions, and when one is offered as
evidence in an investigation or to an assessor.

## Procedure

**1 — Define answerable as a set of fields.** Acting identity, principal,
delegation chain, scopes exercised, and whether the run is replayable. Fewer
than all of them and some question is unanswerable; say which.

**2 — Score a complete record against the definition.** This is the good case
and it should pass — establishing that the definition is achievable rather than
aspirational.

**3 — Construct the impersonation case.** A token where the actor equals the
subject rather than being nested under it. The record is complete and consistent
and the attribution is wrong. Show that it scores identically on completeness.

**4 — Add the check that separates them.** Delegation has a distinct actor claim
nested under the subject; impersonation does not. The record must carry that
distinction, or completeness is all you can ever measure.

**5 — Test the no-replay case.** A record whose run cannot be replayed answers
what happened and never why. Mark it partially answerable rather than
answerable.

## Output contract

```json
{
  "definition": {"fields": ["str"], "replayable_required": true},
  "cases": [{"name": "complete|impersonated|no_replay", "fields_present": ["str"],
             "consistent": true, "answerable": false, "why": "str"}],
  "distinguisher": {"check": "str", "present": false}
}
```

## Failure modes

- **Measuring completeness.** The false record is complete.
- **Accepting a matching subject as delegation.** Impersonation matches too.
- **Calling a non-replayable record answerable.** It answers what, not why.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/regulatory/autonomous-action-auditability/scripts/autonomous_action_auditability.py
SCRIPT = "skills/regulatory/autonomous-action-auditability/scripts/autonomous_action_auditability.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The complete record names the acting identity, principal, chain and scopes and is replayable, so it is answerable. Impersonation produces a complete, consistent and false record attributing the merge to the human. Missing replay fields break the other half. The drill reports 1 of 3 sampled actions fully answerable.

## Your turn

Run the drill on three real production actions from last week. The field you cannot fill is your auditability gap, stated precisely — and a number like "1 of 3" is far more useful to a supervisor than a paragraph about comprehensive logging.

---

**Next → [E2.9 · Regulator and auditor conversations](https://spbreed.github.io/cyber-commons/lessons/E2.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*